# 174 — World models y simulación interna

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**World model**: el agente aprende un modelo del entorno y planifica *dentro* de
él. Arquitectura V-M-C (Ha & Schmidhuber, arXiv:1803.10122):

- **V** comprime la observación en un latente `z_t` (VAE).
- **M** aprende la dinámica estocástica `P(z_{t+1} | z_t, a_t)` (MDN-RNN).
- **C** es una política pequeña sobre `[z_t, h_t]`, entrenable **en el sueño**.

**Por qué latente y no píxeles**: predecir apariencia modela detalle irrelevante
y promedia futuros en imágenes borrosas. **JEPA** predice representaciones de lo
faltante; **Dreamer** entrena actor-crítico con rollouts imaginados de ~15 pasos
(DreamerV3: 150+ tareas con hiperparámetros fijos).

**Riesgo central — explotación del modelo**: el planificador encuentra los
errores del modelo y los "aprovecha". Mitigación: horizonte corto, ensembles,
re-planificación frecuente.


## 🧮 Ejemplo clave (imaginación tabular)

Estados {A,B,C}, recompensa 1 al entrar en C, dinámica aprendida:
`avanzar: A→B 0.9; B→C 0.8` · `saltar: A→C 0.4; B→C 0.5`.

Desde A con horizonte 2, retorno imaginado:
plan [avanzar, avanzar] = 0.9·0.8 = **0.72**; plan [saltar, saltar] =
0.4 + 0.6·0.4 = **0.64** → gana avanzar, sin ejecutar nada real. Pero si la
transición real B→C fuera 0.5 (no 0.8), el plan ganador rendiría 0.45 en el
mundo: eso es *model exploitation*.

El laboratorio de abajo aplica el mismo escepticismo a las afirmaciones de
frontera: separar lo que el modelo (o el paper) afirma de lo verificado.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("frontier", seed=174)
show(result)


## Reflexión

1. En el ejemplo tabular, ¿qué dato del modelo habría que perturbar para que el
   plan [saltar, saltar] pasara a ser el mejor, y cuánto?
2. ¿Por qué predecir representaciones (JEPA) evita el problema de los futuros
   borrosos que sufre la predicción de píxeles?
3. "Nuestro generador de video entiende la física": ¿qué experimento de
   intervención pedirías antes de aceptar esa afirmación como `current` en el
   contrato del laboratorio?
